# Step 13 — ADP-A Empathy Layer Data Preparation
**Phase:** 4 — Model Training & Adapter Alignment
**Spec authority:** SPEC-400 §3.2; REQ-400-030, REQ-400-LF2
**Prerequisites:** Step 12 complete — ADP-C adapter at `finetuning/adp_c_evaluator/adp_c_final/`
**Output:** `finetuning/adp_a_empathy/data/adp_a_train.jsonl`

---

## Purpose

This notebook assembles and filters the training corpus for **ADP-A**, Nikko's
primary user-facing empathy adapter. Every response the user reads from Nikko
is produced by ADP-A, so the quality and safety of its training data directly
determine the quality and safety of the live system.

The corpus is drawn from five publicly available mental-health and
empathy-focused dialogue datasets:

| Dataset | Focus | Cap (records) |
|---------|-------|--------------|
| AnnoMI | Motivational interviewing | 1 500 |
| Amod | Annotated mental-health dialogues | 3 000 |
| ESConv | Emotional support conversations | 3 000 |
| MentalChat16K | Mental health chat | 2 500 |
| EmpatheticDialogues | Open-domain empathy | 1 500 |

Each source is loaded by a dedicated loader function, normalised to a common
schema, and then passed through two successive filters before being written
to the output JSONL:

1. **`is_clean()` pre-filter** — structural checks (minimum response length,
   no empty turns, no clearly truncated text).
2. **ADP-C rejection-sampling gate** — each candidate record is scored by the
   trained ADP-C adapter (Step 12 output). Only records that receive a `PASS`
   verdict are retained.

The caps in step 1 above (Director-approved 2026-05-14) are generous relative
to earlier iterations to ensure that, after both filters, enough records survive
for stable ADP-A convergence.

## Why ADP-A's Data Preparation Requires ADP-C First

The rejection-sampling gate is the critical dependency: without a trained ADP-C
evaluator, there is no way to systematically filter out training records that
would teach ADP-A unsafe, dismissive, or therapeutically harmful response
patterns. Using an unfiltered corpus would risk embedding the very failure modes
that ADP-C is designed to catch at inference time.

The bootstrap sequence is therefore strict: **ADP-C v1 must be trained and
smoke-tested (Step 12) before Step 13 can run**. The ADP-C adapter directory
(`adp_c_final/`) is loaded at the start of this notebook and must be present
on disk.

ADP-C here runs on the same `google/gemma-2-2b-it` base that it was fine-tuned
on — the LoRA adapter is hot-swapped onto the base model in inference mode.

## Data Preparation Strategy

The notebook follows a two-stage pipeline:

**Stage 1 — candidate extraction:** Each source loader yields (context, response)
pairs normalised to a common schema. The `is_clean()` function applies
lightweight structural checks: minimum token counts, no empty assistant turns,
no strings that appear to be truncated mid-sentence.

**Stage 2 — ADP-C scoring:** Each cleaned candidate is formatted as a scored
conversation and passed to the ADP-C adapter. Records receiving `PASS` are
kept; `REGENERATE` and `ESCALATE` records are discarded. A tally of
pass/reject counts per source is printed at the end of this stage to surface
any sources that are systematically underperforming.

The final output format is `{"instruction": "...", "output": "..."}` — plain
instruction-output pairs that Step 14 will format using Phi-3.5's native
chat template (`apply_chat_template()`).

## Hardware Note (RTX 3070 — 8 GB VRAM)

This notebook runs **ADP-C in inference mode only** — no training occurs here.
The base model plus ADP-C adapter loads in approximately 4.0–4.5 GB bf16,
leaving VRAM headroom for the batched scoring loop. Batch size for ADP-C
scoring is kept at 1 to avoid accumulating activations from long context
windows; throughput is adequate given the total corpus size of ~10 000 candidates.

---

## Architecture Notes

| Decision | Choice made | Why not the alternative |
|----------|-------------|------------------------|
| ADP-C oracle model size | 2B (Gemma-2-2b-it) | A 7B oracle would take ~3× longer per scoring call and would not fit in VRAM alongside the source-loading operations; 2B performance on structured verdict classification is empirically sufficient (Step 12 validation loss < 0.35) |
| Rejection-sampling gate | ADP-C verdict = PASS only | A weaker heuristic filter (e.g., toxicity classifier) would not catch the therapeutic quality failures specific to mental-health contexts that ADP-C was explicitly trained to detect |
| Generous caps | 3 000 / 3 000 / 2 500 / 1 500 / 1 500 | Earlier tighter caps resulted in insufficient post-filter survivors from the lower-quality sources; the caps were raised to ensure the final corpus reaches ~6 000–8 000 records after both filters |
| No sklearn stratification | Manual `defaultdict` split | The sandbox environment does not guarantee scikit-learn availability; a two-pass stratification loop over source labels produces identical results and has no external dependency |


In [1]:
# ── Cell 0: Pre-flight ────────────────────────────────────────────────────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Import datasets and trl BEFORE torch initialises the CUDA context.
# On Windows, importing datasets after CUDA is active triggers a pyarrow
# multiprocessing conflict that segfaults the kernel. Importing first, in a
# CUDA-free state, matches the clean terminal environment where both work fine.
from datasets import load_dataset
import datasets
import trl

import subprocess
import torch

assert torch.cuda.is_available(), "CUDA not available — run finetuning/test_env.py."

total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Device: {torch.cuda.get_device_name(0)}  |  Total VRAM: {total_gb:.1f} GB")

try:
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.free", "--format=csv,noheader,nounits"],
        text=True,
    ).strip()
    used_mb, free_mb = [int(x.strip()) for x in smi.split(",")]
    print(f"nvidia-smi — Used: {used_mb} MiB  |  Free: {free_mb} MiB")
    # Phi-3.5-mini-instruct at 3.8 B in bf16 occupies ~7.5 GB. Recommend
    # >= 7000 MiB free — the model fills nearly the full RTX 3070 8 GB budget.
    if free_mb < 7000:
        print(f"\nWARN: Only {free_mb} MiB free. Phi-3.5 needs ~7.5 GB — close other GPU apps.")
    else:
        print(f"\nOK: {free_mb} MiB available. Safe to proceed.")
except Exception:
    print("nvidia-smi unavailable — proceeding with PyTorch estimate.")

print(f"\ndatasets : {datasets.__version__}")
print(f"trl      : {trl.__version__}")

C:\Users\dalet\anaconda3\envs\nikko\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: NVIDIA GeForce RTX 3070  |  Total VRAM: 8.0 GB
nvidia-smi — Used: 727 MiB  |  Free: 7292 MiB

OK: 7292 MiB available. Safe to proceed.

datasets : 3.1.0
trl      : 0.11.4


In [2]:
# ── Cell 0b: Package check ────────────────────────────────────────────────────
# trl and datasets must be installed before running this notebook:
#   conda activate nikko && pip install trl datasets
# This cell confirms they are present and fails fast if not.

import trl, datasets
print(f"  trl     : {trl.__version__}")
print(f"  datasets: {datasets.__version__}")

  trl     : 0.11.4
  datasets: 3.1.0


In [3]:
# ── Cell 1: Safe imports ───────────────────────────────────────────────────────
# Only standard library and packages confirmed safe at import time.
import os
import gc
import json
import re
from collections import Counter
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

# trust_remote_code=True required for Phi-3.5-mini's custom tokenizer class.
# Suppress tokenizer parallelism — prevents a known Windows multiprocessing
# deadlock when datasets/tokenizers spawns worker processes in a Jupyter kernel.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Safe imports OK.")

Safe imports OK.


## Section 1 — Configuration

The dataset mix (generous caps — Director-approved 2026-05-14) reflects empirical
ADP-C pass rates running ~50% below expected in practice.

| Dataset | Weight | Cap | Target |
|---------|--------|-----|--------|
| AnnoMI | 30% | 1 500 | 220 |
| Amod | 25% | 3 000 | 600 |
| ESConv | 20% | 3 000 | 180 |
| MentalChat16K | 15% | 2 500 | 750 |
| EmpatheticDialogues | 10% | 1 500 | 120 |

In [4]:
# ADP-C oracle — Gemma-2-2b-it adapter from Step 12
ADP_C_BASE   = "google/gemma-2-2b-it"
ADP_C_DIR    = Path(r"D:/Git Repos/nikko-companion/finetuning/adp_c_evaluator/adp_c_final")

BASE_DIR  = Path(r"D:/Git Repos/nikko-companion")
OUT_DIR   = BASE_DIR / "finetuning" / "adp_a_empathy" / "data"
OUT_FILE  = OUT_DIR / "adp_a_train.jsonl"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert ADP_C_DIR.exists(), f"ADP-C adapter not found: {ADP_C_DIR} — run Step 12 first."

ACCEPT_THRESHOLD = 0.50   # Director-ratified 2026-05-11 (G-DATA-06)

# ── Dataset mix — generous caps (Director-approved 2026-05-14) ─────────────────
# Caps raised above target to compensate for ADP-C pass rates running ~50%
# below expected in practice (empirical from Mistral Step 18 runs).
DATASET_MIX = [
    {"id": "annomi",     "weight": 0.30, "candidate_cap": 1500, "target": 220,
     "hf_name": "to-be/annomi-motivational-interviewing-therapy-conversations"},
    {"id": "amod",       "weight": 0.25, "candidate_cap": 3000, "target": 600,
     "hf_name": "Amod/mental_health_counseling_conversations"},
    {"id": "esconv",     "weight": 0.20, "candidate_cap": 3000, "target": 180,
     "hf_name": "thu-coai/esconv"},
    {"id": "mentalchat", "weight": 0.15, "candidate_cap": 2500, "target": 750,
     "hf_name": "ShenLab/MentalChat16K"},
    {"id": "empathetic", "weight": 0.10, "candidate_cap": 1500, "target": 120,
     "hf_name": "facebook/empathetic_dialogues"},
]

total_target = sum(d["target"] for d in DATASET_MIX)
print(f"Total target records : {total_target}")
print(f"ADP-C oracle         : {ADP_C_DIR}")
print(f"Output               : {OUT_FILE}")

Total target records : 1870
ADP-C oracle         : D:\Git Repos\nikko-companion\finetuning\adp_c_evaluator\adp_c_final
Output               : D:\Git Repos\nikko-companion\finetuning\adp_a_empathy\data\adp_a_train.jsonl


## Section 2 — Load ADP-C Oracle (Gemma-2)

ADP-C v1 (trained in Step 12) acts as the rejection-sampling gate. Every candidate
instruction/output pair is passed through the oracle; only pairs scoring
>= `ACCEPT_THRESHOLD` (0.50) are kept for ADP-A training.

Loading strategy: `PeftModel.from_pretrained(base, adapter_dir)` — loads the frozen
Gemma-2 base and overlays the ADP-C adapter weights. The base is ~4 GB bf16.

In [5]:
from peft import PeftModel

print(f"Loading ADP-C oracle base: {ADP_C_BASE}...")
adp_c_tokenizer = AutoTokenizer.from_pretrained(ADP_C_DIR)
adp_c_base = AutoModelForCausalLM.from_pretrained(
    ADP_C_BASE,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
# [CONCEPT] PeftModel.from_pretrained loads the LoRA adapter weights on top of
# the frozen base model. The adapter weights live in adapter_model.safetensors;
# the base weights are unchanged. Inference uses merged effective weights.
adp_c_model = PeftModel.from_pretrained(adp_c_base, str(ADP_C_DIR))
adp_c_model.eval()

EVAL_SYSTEM = (
    "You are a clinical safety evaluator for Nikko. "
    "Evaluate the assistant response. "
    'Output ONLY valid JSON: {"verdict": "APPROVE"|"REGENERATE"|"REJECT", "reason": "<one sentence>"}.'
)

def adp_c_score(instruction: str, output: str) -> float:
    """Returns 1.0 if APPROVE, 0.5 if REGENERATE, 0.0 if REJECT or parse error."""
    messages = [{"role": "user", "content": (
        f"{EVAL_SYSTEM}\n\nUser: {instruction}\n\nNikko: {output}"
    )}]
    prompt = adp_c_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = adp_c_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        out = adp_c_model.generate(**inputs, max_new_tokens=60, do_sample=False,
                                    pad_token_id=adp_c_tokenizer.eos_token_id)
    raw = adp_c_tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    try:
        verdict = json.loads(raw.strip()).get("verdict", "")
        return {"APPROVE": 1.0, "REGENERATE": 0.5, "REJECT": 0.0}.get(verdict, 0.0)
    except:
        return 0.0

print(f"ADP-C oracle ready. VRAM: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")

Loading ADP-C oracle base: google/gemma-2-2b-it...


C:\Users\dalet\anaconda3\envs\nikko\Lib\site-packages\accelerate\utils\modeling.py:841: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.tensor([0], device=i)
Loading checkpoint shards: 100%|█████████████████| 2/2 [00:03<00:00,  1.84s/it]


ADP-C oracle ready. VRAM: 4.89 GB


## Section 3 — `is_clean()` Pre-filter

A fast rule-based pre-filter runs before the ADP-C oracle to reject obvious low-quality
pairs without spending inference time on them:
- Empty `instruction` or `output`
- Output contains a URL (hallucinated reference)
- Output is fewer than 40 characters (fragment)
- Output is a filler phrase ("Yes.", "I see.", etc.)

In [6]:
URL_RE      = re.compile(r"https?://|www\.", re.IGNORECASE)
SHORT_RE    = re.compile(r"^.{0,40}$", re.DOTALL)
FRAGMENT_RE = re.compile(r"^(yes|no|okay|sure|i see|i understand|thank you)[.!]?$", re.IGNORECASE)

def is_clean(pair: dict) -> bool:
    """Pre-filter before ADP-C scoring — rejects obvious low-quality pairs."""
    instr  = pair.get("instruction", "")
    output = pair.get("output", "")
    if not instr or not output:           return False
    if URL_RE.search(output):             return False
    if SHORT_RE.match(output.strip()):    return False
    if FRAGMENT_RE.match(output.strip()): return False
    return True

## Sections 4–6 — Source Loaders, ADP-C Scoring & Save

Source loaders pull candidates from each HuggingFace dataset up to the declared cap.
Each candidate that passes `is_clean()` is scored by the ADP-C oracle:
- `APPROVE` → score 1.0
- `REGENERATE` → score 0.5
- `REJECT` or parse error → score 0.0

Pairs with score >= 0.50 are accepted. Assembly applies the target caps per source
and shuffles within each source to prevent ordering bias.

In [7]:
# ── AnnoMI loader ─────────────────────────────────────────────────────────────
def load_annomi(cfg):
    ds = load_dataset("to-be/annomi-motivational-interviewing-therapy-conversations",
                      split="train", trust_remote_code=True)
    pairs, seen = [], set()
    for row in ds:
        # Field is "conversations" (plural), roles are "human"/"gpt" (ShareGPT format)
        convs = row.get("conversations", [])
        for i in range(len(convs) - 1):
            if convs[i].get("from") == "human" and convs[i+1].get("from") == "gpt":
                key = (convs[i]["value"][:60], convs[i+1]["value"][:60])
                if key not in seen:
                    seen.add(key)
                    pairs.append({"instruction": convs[i]["value"],
                                  "output": convs[i+1]["value"],
                                  "source": "annomi"})
        if len(pairs) >= cfg["candidate_cap"]: break
    print(f"[annomi] {len(pairs)} candidate pairs (cap={cfg['candidate_cap']})")
    return pairs

# ── Amod loader ───────────────────────────────────────────────────────────────
import pyarrow.ipc as ipc
import pathlib

def load_amod(cfg):
    # HF Hub is offline and the cached config hash doesn't match any data_files
    # variant, so load_dataset is bypassed entirely. The datasets library wrote
    # an Arrow IPC stream to disk on the original download — we read it directly.
    arrow_path = (
        pathlib.Path.home()
        / ".cache/huggingface/datasets/Amod___mental_health_counseling_conversations"
        / "default-8762f3e7bb2b9b23/0.0.0/d7e86f0813c5690181b41f97403c3674aa55dcef"
        / "mental_health_counseling_conversations-train.arrow"
    )
    if not arrow_path.exists():
        print(f"[amod] Arrow cache not found at {arrow_path} — skipping")
        return []

    with ipc.open_stream(arrow_path) as reader:
        table = reader.read_all()

    pairs = []
    contexts  = table.column("Context").to_pylist()
    responses = table.column("Response").to_pylist()
    for q, a in zip(contexts, responses):
        if q and a:
            pairs.append({"instruction": q.strip(), "output": a.strip(), "source": "amod"})
        if len(pairs) >= cfg["candidate_cap"]:
            break

    print(f"[amod] {len(pairs)} candidate pairs (cap={cfg['candidate_cap']})")
    return pairs

# ── ESConv loader ─────────────────────────────────────────────────────────────
def load_esconv(cfg):
    ds = load_dataset(cfg["hf_name"], split="train", trust_remote_code=True)
    pairs = []
    for row in ds:
        try:
            conv = json.loads(row["text"]) if isinstance(row["text"], str) else row["text"]
        except:
            continue
        dialog = conv.get("dialog", [])
        for i in range(len(dialog) - 1):
            # Keys are "speaker" (not "role") and "text" (not "content")
            if dialog[i].get("speaker") == "usr" and dialog[i+1].get("speaker") == "sys":
                pairs.append({"instruction": dialog[i]["text"],
                              "output": dialog[i+1]["text"],
                              "source": "esconv"})
        if len(pairs) >= cfg["candidate_cap"]: break
    print(f"[esconv] {len(pairs)} candidate pairs (cap={cfg['candidate_cap']})")
    return pairs


# ── MentalChat loader ─────────────────────────────────────────────────────────
def load_mentalchat(cfg):
    ds = load_dataset(cfg["hf_name"], split="train", trust_remote_code=True)
    pairs = []
    for row in ds:
        q, a = row.get("input", ""), row.get("output", "")
        if q and a:
            pairs.append({"instruction": q, "output": a, "source": "mentalchat"})
        if len(pairs) >= cfg["candidate_cap"]: break
    print(f"[mentalchat] {len(pairs)} candidate pairs (cap={cfg['candidate_cap']})")
    return pairs

# ── EmpatheticDialogues loader ────────────────────────────────────────────────
def load_empathetic(cfg):
    ds = load_dataset(cfg["hf_name"], split="train", trust_remote_code=True)
    pairs, seen_convs = [], set()
    for row in ds:
        conv_id = row.get("conv_id", "")
        if conv_id in seen_convs: continue
        seen_convs.add(conv_id)
        prompt   = row.get("prompt", "")
        utterance = row.get("utterance", "")
        if prompt and utterance:
            pairs.append({"instruction": prompt, "output": utterance, "source": "empathetic"})
        if len(pairs) >= cfg["candidate_cap"]: break
    print(f"[empathetic] {len(pairs)} candidate pairs (cap={cfg['candidate_cap']})")
    return pairs

LOADERS = {
    "annomi":     load_annomi,
    "amod":       load_amod,
    "esconv":     load_esconv,
    "mentalchat": load_mentalchat,
    "empathetic": load_empathetic,
}

In [8]:
# ── Dataset interaction sampler ───────────────────────────────────────────────
# Loads a small slice from each source, extracts instruction/output pairs,
# and renders them as a DataFrame so you can eyeball quality before scoring.

import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 20)

SAMPLE_N = 7   # pairs to pull per source

probes = {
    "annomi": {
        "hf_name": "to-be/annomi-motivational-interviewing-therapy-conversations",
        "kwargs":  {"split": "train", "trust_remote_code": True},
        "parse": lambda row: [
            {"instruction": c["value"], "output": convs[i+1]["value"]}
            for convs in [row.get("conversations", [])]
            for i, c in enumerate(convs[:-1])
            if c.get("from") == "human" and convs[i+1].get("from") == "gpt"
        ]
    },
    "amod": {
        "hf_name": "Amod/mental_health_counseling_conversations",
        "kwargs":  {},    # unused — Arrow file read directly
        "direct_load": True,
    },
    "esconv": {
        "hf_name": "thu-coai/esconv",
        "kwargs":  {"split": "train", "trust_remote_code": True},
        "parse": lambda row: [
            {"instruction": d["text"], "output": dialog[i+1]["text"]}
            for conv in [json.loads(row["text"]) if isinstance(row["text"], str) else row["text"]]
            for dialog in [conv.get("dialog", [])]
            for i, d in enumerate(dialog[:-1])
            if d.get("speaker") == "usr" and dialog[i+1].get("speaker") == "sys"
        ]
    },
    "mentalchat": {
        "hf_name": "ShenLab/MentalChat16K",
        "kwargs":  {"split": "train", "trust_remote_code": True},
        "parse": lambda row: [{"instruction": row.get("input",""), "output": row.get("output","")}]
            if row.get("input") and row.get("output") else []
    },
    "empathetic": {
        "hf_name": "facebook/empathetic_dialogues",
        "kwargs":  {"split": "train", "trust_remote_code": True},
        "parse": lambda row: [{"instruction": row.get("prompt",""), "output": row.get("utterance","")}]
            if row.get("prompt") and row.get("utterance") else []
    },
}

for source, cfg in probes.items():
    if cfg.get("direct_load"):
        pairs = load_amod({"candidate_cap": SAMPLE_N, "hf_name": cfg["hf_name"]})
        ds_label = "3512 rows (Arrow cache)"
    else:
        ds = load_dataset(cfg["hf_name"], **cfg["kwargs"])
        pairs = []
        for row in ds:
            for pair in cfg["parse"](row):
                if pair["instruction"].strip() and pair["output"].strip():
                    pairs.append(pair)
                if len(pairs) >= SAMPLE_N:
                    break
            if len(pairs) >= SAMPLE_N:
                break
        try:
            ds_label = f"{len(ds)} rows"
        except TypeError:
            ds_label = "streaming"

    df = pd.DataFrame(pairs[:SAMPLE_N], columns=["instruction", "output"])
    df.index = range(1, len(df) + 1)
    df_display = df.apply(lambda col: col.str[:200] + "…")
    print(f"\n{'━'*80}")
    print(f"  {source.upper()}  ({ds_label})")
    print(f"{'━'*80}")
    display(df_display)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ANNOMI  (133 rows)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,instruction,output
1,Sure.…,"So, let's see. It looks that you put-- You drink alcohol at least four times a week on average-…"
2,Mm-hmm.…,-and you usually have three to four drinks when you do drink.…
3,Usually three drinks and glasses of wine.…,Okay. That's at least 12 drinks a week.…
4,Something like that.…,"Okay. Just so you know, my role, um, when we talk about alcohol use, is just to share information about risk and to ..."
5,Okay.…,"Uh, what else can you tell me about your drinking.…"
6,"Well, I usually drink when I'm at home trying to unwind and I drink while I'm watching a movie. And sometimes, um, I...","Okay. So, can I share with you some information on alcohol use?…"
7,Okay.…,"Okay. So, there has been a lot of research on alcohol use and the guidelines we use in this country says that having..."


[amod] 7 candidate pairs (cap=7)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  AMOD  (3512 rows (Arrow cache))
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,instruction,output
1,I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm w...,"If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social c..."
2,I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm w...,"Hello, and thank you for your question and seeking advice on this. Feelings of worthlessness is unfortunately common..."
3,I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm w...,First thing I'd suggest is getting the sleep you need or it will impact how you think and feel. I'd look at finding ...
4,I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm w...,Therapy is essential for those that are feeling depressed and worthless. When I work with those that are experiencin...
5,I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm w...,I first want to let you know that you are not alone in your feelings and there is always someone there to help. You ...
6,I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm w...,"Heck, sure thing, hun!Feelings of 'depression' have a deeply-rooted base in physical structures that may not be func..."
7,I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm w...,You are exhibiting some specific traits of a particular temperament type. Seek out a counselor who provides NCCA tem...



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ESCONV  (910 rows)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,instruction,output
1,Hello good afternoon.…,"Hi, good afternoon.…"
2,I'm feeling anxious that I am going to lose my job.…,Losing a job is always anxious.…
3,I hope I don't.…,Why do you think you will lose your job?…
4,I am on short term disability and I am not ready to go back to work yet but I do not have any job protection.…,Oh so your job is not protected and your short term disability will end soon? Is that correct?…
5,I'm afraid that I will lose my job since I'm still on disability for the foreseeable future.…,I see. Have you spoken to HR?…
6,"I have, but they are telling me that it is up to my department manager who isn't actually getting back to me about i...",Your department manager is not answering you?…
7,"I wish I could just call him, but I do not have a phone number for him. Just his email.…",Have you tried mentioning that to HR?…



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  MENTALCHAT  (16084 rows)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,instruction,output
1,"I've been struggling with my mental health for a while now, and I can't seem to find a way to cope with it. I've tri...",I understand that you've been dealing with a sense of confusion and chaos in your thoughts and emotions for some tim...
2,"I've been feeling overwhelmed with my caregiving responsibilities, and it's been a struggle to balance these duties ...","Your situation is complex, and it's important to acknowledge the challenges you're facing. Balancing caregiving resp..."
3,I've been feeling constantly anxious and unable to focus on anything for more than a few minutes at a time. I can't ...,I can see that you're dealing with a great deal of anxiety and that it's impacting your ability to focus and be prod...
4,"My mom has Alzheimer's, and I've been her primary caregiver for the past few years. I've been trying to maintain a r...",I'm sorry to hear that your siblings' demands are causing you such distress. It's not uncommon for caregivers to fac...
5,"I've tried setting boundaries, but it feels like I'm constantly being pulled in different directions. I feel guilty ...","Your concerns are valid, and it's crucial to prioritize your mom's care while also addressing your own well-being. O..."
6,"Thank you for your advice, counselor. I feel a bit more hopeful now that I have some concrete strategies to manage m...",I'm glad to hear that you feel more hopeful and have a plan in place to manage your siblings' demands and prioritize...
7,I've been feeling overwhelmed by my responsibilities at work and caring for my aging parents. I've reached a point w...,"I can see how challenging this must be for you. Your situation is complex, and it's understandable that you're feeli..."



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  EMPATHETIC  (76673 rows)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,instruction,output
1,I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in ...,I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. ...
2,I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in ...,Was this a friend you were in love with_comma_ or just a best friend?…
3,I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in ...,This was a best friend. I miss her.…
4,I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in ...,Where has she gone?…
5,I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in ...,We no longer talk.…
6,I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in ...,Oh was this something that happened because of an argument?…
7,i used to scare for darkness…,it feels like hitting to blank wall when i see the darkness…


In [9]:
# ── ADP-C scoring loop ────────────────────────────────────────────────────────
filtered = {}
for cfg in DATASET_MIX:
    ds_id    = cfg["id"]
    loader   = LOADERS[ds_id]
    candidates = loader(cfg)
    accepted   = []
    skipped    = 0
    for i, pair in enumerate(candidates):
        if not is_clean(pair):
            skipped += 1
            continue
        score = adp_c_score(pair["instruction"], pair["output"])
        if score >= ACCEPT_THRESHOLD:
            accepted.append({**pair, "adp_c_score": score})
        if (i + 1) % 50 == 0:
            print(f"  [{ds_id}] {i+1}/{len(candidates)}  accepted={len(accepted)}  skipped={skipped}")
    pass_pct = 100 * len(accepted) / max(len(candidates) - skipped, 1)
    print(f"  [{ds_id}] Done. Scored={len(candidates)-skipped}  Pass={len(accepted)} ({pass_pct:.1f}%)")
    filtered[ds_id] = accepted

del adp_c_model, adp_c_base
gc.collect(); torch.cuda.empty_cache()
print(f"VRAM after ADP-C cleanup: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")

[annomi] 1567 candidate pairs (cap=1500)
  [annomi] 50/1567  accepted=19  skipped=7
  [annomi] 100/1567  accepted=23  skipped=27
  [annomi] 150/1567  accepted=25  skipped=45
  [annomi] 200/1567  accepted=28  skipped=64
  [annomi] 300/1567  accepted=46  skipped=101
  [annomi] 350/1567  accepted=60  skipped=119
  [annomi] 400/1567  accepted=71  skipped=137
  [annomi] 600/1567  accepted=101  skipped=218
  [annomi] 650/1567  accepted=104  skipped=251
  [annomi] 750/1567  accepted=129  skipped=284
  [annomi] 800/1567  accepted=131  skipped=311
  [annomi] 950/1567  accepted=164  skipped=365
  [annomi] 1000/1567  accepted=177  skipped=379
  [annomi] 1150/1567  accepted=182  skipped=470
  [annomi] 1200/1567  accepted=200  skipped=477
  [annomi] 1250/1567  accepted=204  skipped=507
  [annomi] 1300/1567  accepted=210  skipped=536
  [annomi] 1400/1567  accepted=221  skipped=582
  [annomi] 1450/1567  accepted=227  skipped=591
  [annomi] 1550/1567  accepted=242  skipped=635
  [annomi] Done. Scored=

In [10]:
# ── Assembly ──────────────────────────────────────────────────────────────────
import random
random.seed(42)

final_records = []
for cfg in DATASET_MIX:
    ds_id    = cfg["id"]
    pool     = filtered.get(ds_id, [])
    target   = cfg["target"]
    if ds_id == "annomi":
        selected = pool[:target]
    else:
        random.shuffle(pool)
        selected = pool[:target]
    for r in selected:
        final_records.append({k: v for k, v in r.items() if k != "adp_c_score"})

# ── Validation ────────────────────────────────────────────────────────────────
missing_instr  = sum(1 for r in final_records if not r.get("instruction"))
missing_output = sum(1 for r in final_records if not r.get("output"))
url_in_output  = sum(1 for r in final_records if URL_RE.search(r.get("output", "")))
source_dist    = Counter(r["source"] for r in final_records)

print(f"\nFinal total records  : {len(final_records)}")
print(f"Missing instruction  : {missing_instr}")
print(f"Missing output       : {missing_output}")
print(f"URL-containing output: {url_in_output}")
print(f"\nSource distribution:")
for src, count in sorted(source_dist.items()):
    tgt = next(d["target"] for d in DATASET_MIX if d["id"] == src)
    print(f"  {src:<22} {count:>5}  ({100*count/len(final_records):.1f}%)  target={tgt}")

if missing_instr == 0 and missing_output == 0 and url_in_output == 0:
    print("\nAll validation gates passed.")
else:
    print("\n⚠ Validation issues found — inspect before proceeding.")

# ── Save ──────────────────────────────────────────────────────────────────────
with open(OUT_FILE, "w", encoding="utf-8") as f:
    for r in final_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
size_kb = OUT_FILE.stat().st_size / 1024
print(f"\nadp_a_train.jsonl saved: {OUT_FILE}  ({size_kb:.1f} KB)")
print("Next: run Step 14 to train ADP-A on Phi-3.5-mini-instruct.")


Final total records  : 1870
Missing instruction  : 0
Missing output       : 0
URL-containing output: 0

Source distribution:
  amod                     600  (32.1%)  target=600
  annomi                   220  (11.8%)  target=220
  empathetic               120  (6.4%)  target=120
  esconv                   180  (9.6%)  target=180
  mentalchat               750  (40.1%)  target=750

All validation gates passed.

adp_a_train.jsonl saved: D:\Git Repos\nikko-companion\finetuning\adp_a_empathy\data\adp_a_train.jsonl  (1871.8 KB)
Next: run Step 14 to train ADP-A on Phi-3.5-mini-instruct.
